# Problem 1

In [1]:
def simulate_fibonacci_cascade(x_start, y_start):
    """
    Simulates the unrolled chemical reaction cascade to compute 
    the 12th step of a Fibonacci-like sequence.
    """
    # Initialize our molecular "buckets" (concentrations)
    species = {
        'x': x_start, 'y': y_start, 
        'a': 0, 'b': 0, 'c': 0, 'd': 0, 'e': 0, 'f': 0, 
        'g': 0, 'h': 0, 'i': 0, 'j': 0, 'k': 0
    }

    # Helper function to model a stoichiometric fanout reaction completing 100%.
    # The reactant is fully consumed, and the products gain that exact amount.
    def fire_reaction(reactant, prod1, prod2=None):
        amount = species[reactant]
        if amount > 0:
            species[reactant] = 0 # Reactant is fully consumed
            species[prod1] += amount
            
            if prod2:
                species[prod2] += amount
                print(f"Reaction [{reactant} -> {prod1} + {prod2}] fired {amount} times.")
            else:
                print(f"Reaction [{reactant} -> {prod1}] fired {amount} times.")
                
            # Show the accumulating pools for the next steps
            p2_str = f", {prod2} pool: {species[prod2]}" if prod2 else ""
            print(f"   -> Resulting {prod1} pool: {species[prod1]}{p2_str}")

    print(f"==================================================")
    print(f"Simulating Cascade for Inputs: x = {x_start}, y = {y_start}")
    print(f"==================================================")
    
    # 1. Initialization Phase
    fire_reaction('x', 'a')
    fire_reaction('y', 'a', 'b')
    print("-" * 50)
    
    # 2. The Propagation Cascade (Addition via Stoichiometry)
    fire_reaction('a', 'b', 'c')
    fire_reaction('b', 'c', 'd')
    fire_reaction('c', 'd', 'e')
    fire_reaction('d', 'e', 'f')
    fire_reaction('e', 'f', 'g')
    fire_reaction('f', 'g', 'h')
    fire_reaction('g', 'h', 'i')
    fire_reaction('h', 'i', 'j')
    fire_reaction('i', 'j', 'k')
    print("-" * 50)

    # 3. The Terminator Reaction (The Sink)
    fire_reaction('j', 'k')
    
    print(f"\n>>> FINAL ANSWER: Species 'k' contains {species['k']} molecules <<<\n")
    return species['k']

# ==========================================
# Run Test 1: Standard Fibonacci Sequence
# Expected 12th step: 144
# ==========================================
simulate_fibonacci_cascade(0, 1)

# ==========================================
# Run Test 2: Custom Sequence
# Expected 12th step: 1275
# ==========================================
simulate_fibonacci_cascade(3, 7)

Simulating Cascade for Inputs: x = 0, y = 1
Reaction [y -> a + b] fired 1 times.
   -> Resulting a pool: 1, b pool: 1
--------------------------------------------------
Reaction [a -> b + c] fired 1 times.
   -> Resulting b pool: 2, c pool: 1
Reaction [b -> c + d] fired 2 times.
   -> Resulting c pool: 3, d pool: 2
Reaction [c -> d + e] fired 3 times.
   -> Resulting d pool: 5, e pool: 3
Reaction [d -> e + f] fired 5 times.
   -> Resulting e pool: 8, f pool: 5
Reaction [e -> f + g] fired 8 times.
   -> Resulting f pool: 13, g pool: 8
Reaction [f -> g + h] fired 13 times.
   -> Resulting g pool: 21, h pool: 13
Reaction [g -> h + i] fired 21 times.
   -> Resulting h pool: 34, i pool: 21
Reaction [h -> i + j] fired 34 times.
   -> Resulting i pool: 55, j pool: 34
Reaction [i -> j + k] fired 55 times.
   -> Resulting j pool: 89, k pool: 55
--------------------------------------------------
Reaction [j -> k] fired 89 times.
   -> Resulting k pool: 144

>>> FINAL ANSWER: Species 'k' contains

1275

# Problem 2

# =================================================================
# GROUP 1: Signal Fanout, Delay Input, and Computation (Blue-to-Red)
# =================================================================
# These reactions split the signals into feedforward and feedback paths
g + X  kslow -> A + R1
g + B1 kslow -> F + C + R2
g + B2 kslow -> H + E

# 1/8 Scalar Multipliers (Feedforward to Y)
8A kfast -> Y
8C kfast -> Y
8E kfast -> Y

# 1/8 Scalar Multipliers (Feedback to X)
8F kfast -> X
8H kfast -> X

# =================================================================
# GROUP 2: Delay Operations (Phase Shifting)
# =================================================================
# Red-to-Green Phase
b + R1 kslow -> G1
b + R2 kslow -> G2

# Green-to-Blue Phase
r + G1 kslow -> B1
r + G2 kslow -> B2

# =================================================================
# GROUP 3: Color Concentration (Presence) Indicators
# =================================================================
# Red Phase Indicators
2Y  kfast -> 2Y + R'
2R1 kfast -> 2R1 + R'
2R2 kfast -> 2R2 + R'

# Green Phase Indicators
2G1 kfast -> 2G1 + G'
2G2 kfast -> 2G2 + G'

# Blue Phase Indicators
2X  kfast -> 2X + B'
2B1 kfast -> 2B1 + B'
2B2 kfast -> 2B2 + B'

# Waste/Cleanup of Indicators
2R' kfast -> 0
2G' kfast -> 0
2B' kfast -> 0

# =================================================================
# GROUP 4: Absence Indicators
# =================================================================
# Generation
2Sr kslow -> 2Sr + r
2Sg kslow -> 2Sg + g
2Sb kslow -> 2Sb + b

# Consumption by Presence Indicators
R' + r kfast -> R'
G' + g kfast -> G'
B' + b kfast -> B'

# =================================================================
# GROUP 5: Positive Feedback Kinetics (Pulling)
# =================================================================
# Red pulls Blue-to-Red inputs
R' + X  kfast -> R' + A + R1
R' + B1 kfast -> R' + F + C + R2
R' + B2 kfast -> R' + H + E

# Green pulls Red-to-Green delays
G' + R1 kfast -> G' + G1
G' + R2 kfast -> G' + G2

# Blue pulls Green-to-Blue delays
B' + G1 kfast -> B' + B1
B' + G2 kfast -> B' + B2

In [2]:
def simulate_chemical_biquad_kinetics(inputs):
    # Shared Delay line memory (Blue phase)
    B1, B2 = 0, 0
    
    print("Starting Corrected Stochastic Kinetics Simulation...\n")
    print("-" * 60)
    
    for cycle, X_current in enumerate(inputs, start=1):
        X = int(X_current)
        
        # Intermediate computational species
        A, C, E, F, H = 0, 0, 0, 0, 0
        
        # Output and the "Red" copied delayed species
        Y = 0
        R1, R2 = 0, 0
        
        while True:
            reactions_fired = False
            
            # --- FANOUT / SIGNAL SPLITTING ---
            if B1 > 0:
                B1 -= 1; C += 1; F += 1; R2 += 1
                reactions_fired = True
                
            if B2 > 0:
                B2 -= 1; E += 1; H += 1
                reactions_fired = True
                
            if X > 0:
                X -= 1; A += 1; R1 += 1
                reactions_fired = True

            # --- MULTIPLIERS (FEEDFORWARD TO Y) ---
            if A >= 8:
                A -= 8; Y += 1
                reactions_fired = True
            if C >= 8:
                C -= 8; Y += 1
                reactions_fired = True
            if E >= 8:
                E -= 8; Y += 1
                reactions_fired = True

            # --- MULTIPLIERS (FEEDBACK TO X) ---
            if F >= 8:
                F -= 8; X += 1  # This newly created X feeds back into the loop!
                reactions_fired = True
            if H >= 8:
                H -= 8; X += 1
                reactions_fired = True

            if not reactions_fired:
                break
                
        print(f"CYCLE {cycle} | Input X Molecules = {X_current}")
        print(f"  -> Y Molecules Produced : {Y}")
        
        # Shift the shared delay line for the next cycle
        B2 = R2       
        B1 = R1  
        
        print(f"  -> State Shifted: B1={B1}, B2={B2}")
        print("-" * 60)

input_sequence = [100, 5, 500, 20, 250]
simulate_chemical_biquad_kinetics(input_sequence)

Starting Corrected Stochastic Kinetics Simulation...

------------------------------------------------------------
CYCLE 1 | Input X Molecules = 100
  -> Y Molecules Produced : 12
  -> State Shifted: B1=100, B2=0
------------------------------------------------------------
CYCLE 2 | Input X Molecules = 5
  -> Y Molecules Produced : 14
  -> State Shifted: B1=17, B2=100
------------------------------------------------------------
CYCLE 3 | Input X Molecules = 500
  -> Y Molecules Produced : 78
  -> State Shifted: B1=514, B2=17
------------------------------------------------------------
CYCLE 4 | Input X Molecules = 20
  -> Y Molecules Produced : 76
  -> State Shifted: B1=86, B2=514
------------------------------------------------------------
CYCLE 5 | Input X Molecules = 250
  -> Y Molecules Produced : 114
  -> State Shifted: B1=324, B2=86
------------------------------------------------------------
